In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import to_json

In [3]:
spark = SparkSession.builder.master('local[*]').appName('13').getOrCreate()

- `to_json`: 복잡한 column 구조를 텍스트기반의 json 문자열로 압축
- **언제 필요해??**: spark에서의 결과를 자바스크립트 기반 웹 대시보드에 전달해야 하는 상황이라면, 웹서버는 MapType을 직접 이해하지 못하지만, 표준 json 문자열은 이해할 수 있음 (`to_json`이 가교 역할)

### (1) MapType

In [4]:
map_data = [
    ('Tom',{"height":173.5,"weight":69.8}),
    ('Jane',{"height":163.0,"weight":45.5,"blood_p":110.0})
]
map_schema = ['name','properties']

In [5]:
df = spark.createDataFrame(map_data,map_schema)

In [7]:
df.show(truncate=False)
df.printSchema()

+----+---------------------------------------------------+
|name|properties                                         |
+----+---------------------------------------------------+
|Tom |{weight -> 69.8, height -> 173.5}                  |
|Jane|{weight -> 45.5, blood_p -> 110.0, height -> 163.0}|
+----+---------------------------------------------------+

root
 |-- name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)



- `to_json(col)`: `MapType`으로 저장된 column을 전달하면, 해당 데이터를 string(**표준 json 문자열**)으로 변환하여 반환

In [8]:
df_map_str = df.withColumn('profile',to_json(df.properties))

In [10]:
df_map_str.show(truncate=False)
df_map_str.printSchema()

+----+---------------------------------------------------+----------------------------------------------+
|name|properties                                         |profile                                       |
+----+---------------------------------------------------+----------------------------------------------+
|Tom |{weight -> 69.8, height -> 173.5}                  |{"weight":69.8,"height":173.5}                |
|Jane|{weight -> 45.5, blood_p -> 110.0, height -> 163.0}|{"weight":45.5,"blood_p":110.0,"height":163.0}|
+----+---------------------------------------------------+----------------------------------------------+

root
 |-- name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)
 |-- profile: string (nullable = true)



### (2) StructType

In [11]:
struct_schema = StructType([
    StructField('name',StringType(),True),
    StructField('properties',StructType([
        StructField('height',FloatType(),True),
        StructField('weight',FloatType(),True),
        StructField('blood_p',DoubleType(),True)
    ]))
])

In [12]:
df_struct = spark.createDataFrame(map_data,struct_schema)

In [14]:
df_struct.show(truncate=False)
df_struct.printSchema()

+----+--------------------+
|name|properties          |
+----+--------------------+
|Tom |{173.5, 69.8, NULL} |
|Jane|{163.0, 45.5, 110.0}|
+----+--------------------+

root
 |-- name: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- height: float (nullable = true)
 |    |-- weight: float (nullable = true)
 |    |-- blood_p: double (nullable = true)



In [ ]:
df_struct_str = df_struct.withColumn('profile',to_json(df_struct.properties))
# sub-column이 key로, 저장된 값이 value로 할당되어 string으로 ("key":value 형태)변환

In [16]:
df_struct_str.show(truncate=False)
df_struct_str.printSchema()

+----+--------------------+----------------------------------------------+
|name|properties          |profile                                       |
+----+--------------------+----------------------------------------------+
|Tom |{173.5, 69.8, NULL} |{"height":173.5,"weight":69.8}                |
|Jane|{163.0, 45.5, 110.0}|{"height":163.0,"weight":45.5,"blood_p":110.0}|
+----+--------------------+----------------------------------------------+

root
 |-- name: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- height: float (nullable = true)
 |    |-- weight: float (nullable = true)
 |    |-- blood_p: double (nullable = true)
 |-- profile: string (nullable = true)

